<a href="https://colab.research.google.com/github/mille-s/Acoustic_feature_extraction_patient_clustering/blob/main/2026_Acoustic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pyannote

In [1]:
#@title Install packages
! pip install pydub
! pip install praat-parselmouth
! pip install pyannote-audio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 7.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 10.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 49.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 36.6 MB/s  0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-c

In [2]:
#@title Diarisation Pyannote (First, upload .wav file(s) in folder called "audios_full")
# Adapted from Pyannote GitHub code https://github.com/pyannote/pyannote-audio
from pyannote.audio import Pipeline
from pydub import AudioSegment
import statistics
import glob
import os
import codecs
import json
import sys
import pickle

runtime_type = 'GPU'#@param['CPU', 'GPU']

audio_folder = os.path.join('/content', 'audios_full')
if not os.path.exists(audio_folder):
  os.makedirs(audio_folder)
  print(f'!!!WARNING!!! Upload first your audio files in the {audio_folder} folder just created!')
  sys.exit()

# Create output folder
out_folder = os.path.join('/content', 'audios_split')
if not os.path.exists(out_folder):
  os.makedirs(out_folder)

# 1. Load diarisation pipeline from HuggingFace
# Needs own token
pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-community-1",
  token=""
)

if runtime_type == 'GPU':
  import torch
  pipeline.to(torch.device("cuda"))

# 2. Launch analysis of audio files
audio_file_path_list = glob.glob(os.path.join(audio_folder, '*.wav'))

# Process each full audio file
for audio_path in audio_file_path_list:
  # Create output subfolder for each interview
  patient_id = audio_path.rsplit('.', 1)[0].rsplit('/', 1)[1]
  out_split_files = os.path.join(out_folder, patient_id)
  if not os.path.exists(os.path.join(out_split_files)):
    os.makedirs(out_split_files)

  output = pipeline(audio_path)
# What "output" contains:
# | Field                         | Type                | Meaning                                                                               |
# | ----------------------------- | ------------------- | ------------------------------------------------------------------------------------- |
# | speaker_diarization           | Annotation          | Full diarization timeline, including overlapping speech                               |
# | exclusive_speaker_diarization | Annotation          | Diarization with overlaps removed, intended to align more cleanly with transcription  |
# | speaker_embeddings            | NumPy array or None | One embedding per detected speaker, ordered according to speaker_diarization.labels() |

  # Serialise speaker_diarization and exclusive_speaker_diarization for future use
  with open(os.path.join(out_split_files, 'diarization_full.pickle'), 'wb') as fileOut:
      pickle.dump(output.speaker_diarization, fileOut)
  with open(os.path.join(out_split_files, 'diarization_exclusive.pickle'), 'wb') as fileOut:
      pickle.dump(output.exclusive_speaker_diarization, fileOut)

  # 3. Load audio file with pydub to segment it
  original_audio = AudioSegment.from_wav(audio_path)

  # Dictionaries for storing audio sequences from each speaker
  speaker_audios = {}
  speaker_order_and_gaps = {}

  # 4. Retrieve time stamps of each speaker sequence
  previous_turn_end = 0
  previous_speaker = None
  with codecs.open(os.path.join(out_split_files, 'log_turns.txt'), 'w', 'utf-8') as fo1:
    # for turn, speaker in output.speaker_diarization:
    for turn, speaker in output.exclusive_speaker_diarization:
      # Add speaker to list which will provide the order in which people talked for the first time
      if speaker not in speaker_order_and_gaps:
        speaker_order_and_gaps[speaker] = {}
        speaker_order_and_gaps[speaker]['order'] = len(speaker_order_and_gaps)
        speaker_order_and_gaps[speaker]['gaps'] = []
      # Print dialog turn time stamps
      print(f"{speaker} speaks from {turn.start:.3f}s to {turn.end:.3f}s")
      fo1.write(f"{speaker} speaks from {turn.start:.3f}s to {turn.end:.3f}s\n")
      # Calculate gap between turns
      gap = float(f"{turn.start:.3f}") - float(previous_turn_end)
      # Only check gap and update previous turn end if current turn ended after previous turn
      if float(f'{turn.end:.3f}') > float(previous_turn_end):
        # Only check gap and update speaker if previous turn was uttered by a different speaker
        if not speaker == previous_speaker:
          previous_speaker = speaker
          # Do not use the first gap between recording start and first talking
          if not previous_turn_end == 0:
            # In case of overlapping turns, we can have speaker0 from 2s to 20s and speaker1 from 3s to 4s and/or from 18s to 21s
            # We don't want to take into account overlapping turns for gaps; or should we count a 0 gap? I tried, there's lots of 0s...
            # Also, the diarised turns do not end when the voice ends, but later, so the gaps here are not the real gaps.
            if gap > 0:
              print(f'  Gap = {gap}')
              speaker_order_and_gaps[speaker]['gaps'].append(gap)
        # Update turn end (only now so it can be used above to detect the first turn)
        previous_turn_end = f'{turn.end:.3f}'

      # Convert seconds into milliseconds (required by pydub)
      start_ms = int(turn.start * 1000)
      end_ms = int(turn.end * 1000)

      # Extract audio segment
      segment = original_audio[start_ms:end_ms]

      # Add speaker to dico
      if speaker not in speaker_audios:
        speaker_audios[speaker] = AudioSegment.empty()

      # Add segment to speaker entry in dico
      speaker_audios[speaker] += segment

    # 5. Create separate output file for each speaker
    for speaker, audio_segment in speaker_audios.items():
      # Get the position of the speaker, i.e. if they were the first, second, (etc.) to talk in the conversation.
      speaker_position = str(speaker_order_and_gaps[speaker]['order']).zfill(2)
      # output_filename = os.path.join(out_split_files, f"output_speaker_{speaker_position}.wav")
      output_filename = os.path.join(out_split_files, f"out_{speaker}.wav")
      audio_segment.export(output_filename, format="wav")
      print(f"-> File created : {output_filename}")

    # diarization.write_rttm(foAll)
    # print(speaker_order_and_gaps)

  # Get average gap time for each speaker
  # Note that this may not be usable because pyannote does not segment right after the voice stops.
  # So part of the gap is actually on the audio files.
  for logged_speaker in speaker_order_and_gaps:
    order = speaker_order_and_gaps[logged_speaker]['order']
    avg_gap = round(statistics.mean(speaker_order_and_gaps[logged_speaker]['gaps']), 2)
    print(f'Speaker {order}/{len(speaker_order_and_gaps)}: {avg_gap}s average gap.')
    speaker_order_and_gaps[logged_speaker]['avg_gap'] = avg_gap
  with codecs.open(os.path.join(out_split_files, 'speaker_order_and_gaps.json'), 'w', 'utf-8') as fo3:
    json.dump(speaker_order_and_gaps, fo3, ensure_ascii=False, indent=4)


/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:102: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1913.)
  std = sequences.std(dim=-1, correction=1)


SPEAKER_01 speaks from 1.735s to 5.363s
SPEAKER_00 speaks from 6.916s to 22.913s
  Gap = 1.553
SPEAKER_01 speaks from 23.403s to 30.322s
  Gap = 0.48999999999999844
SPEAKER_00 speaks from 31.368s to 32.245s
  Gap = 1.0459999999999994
SPEAKER_01 speaks from 32.245s to 32.667s
SPEAKER_00 speaks from 32.667s to 32.971s
SPEAKER_01 speaks from 32.971s to 33.157s
SPEAKER_00 speaks from 33.157s to 60.443s
SPEAKER_00 speaks from 60.460s to 60.545s
SPEAKER_00 speaks from 62.215s to 70.602s
SPEAKER_01 speaks from 70.602s to 70.703s
SPEAKER_00 speaks from 70.703s to 71.699s
SPEAKER_01 speaks from 71.699s to 71.969s
SPEAKER_00 speaks from 71.969s to 86.245s
SPEAKER_00 speaks from 87.393s to 88.675s
SPEAKER_01 speaks from 88.675s to 89.890s
SPEAKER_00 speaks from 90.093s to 94.025s
  Gap = 0.20300000000000296
SPEAKER_01 speaks from 94.025s to 95.881s
SPEAKER_00 speaks from 96.168s to 98.345s
  Gap = 0.28700000000000614
SPEAKER_01 speaks from 98.682s to 99.188s
  Gap = 0.3370000000000033
SPEAKER_00 

In [3]:
#@title Praat acoustic analysis of diarized audios (Speaker order, F0 and intra-turn pauses)
# Adapted from Perplexity prompted answers

import parselmouth
from parselmouth.praat import call
import numpy as np
import pandas as pd

def extract_prosodic_feats(wav_path, turn_start=None, turn_end=None):
  snd = parselmouth.Sound(wav_path)

  if turn_start is not None and turn_end is not None:
    snd = snd.extract_part(
        from_time=turn_start,
        to_time=turn_end
    )

  pitch = snd.to_pitch()
  intensity = snd.to_intensity()

  pitch_values = pitch.selected_array["frequency"]
  voiced_pitch_values = pitch_values[pitch_values > 0]

  return {
    "file": wav_path,
    "duration_s": snd.duration,

    # F0 mean / median: Overall pitch level. Confounded by speaker identity and vocal physiology.
    "f0_mean_hz": np.mean(voiced_pitch_values) if len(voiced_pitch_values) else np.nan,
    "f0_median_hz": np.median(voiced_pitch_values) if len(voiced_pitch_values) else np.nan,

    # F0 standard deviation: Pitch variability.	Can reflect intonational movement, but also voicing errors or outliers.
    "f0_std_hz": np.std(voiced_pitch_values) if len(voiced_pitch_values) else np.nan,

    # F0 range (max − min):	Pitch excursion / intonational range.	Sensitive to octave errors and extreme values.
    "f0_min_hz": np.min(voiced_pitch_values) if len(voiced_pitch_values) else np.nan,
    "f0_max_hz": np.max(voiced_pitch_values) if len(voiced_pitch_values) else np.nan,
    "f0_range_hz": np.max(voiced_pitch_values) - np.min(voiced_pitch_values) if len(voiced_pitch_values) else np.nan,

    # Voiced proportion: Percentage of frames that have F0 (not the same as speech ratio: there can be non-silent frames that have no F0)
    "voiced_proportion": len(voiced_pitch_values) / len(pitch.selected_array["frequency"]) if len(voiced_pitch_values) else 0.0,

    # Mean intensity/loudness
    "intensity_mean_db": np.mean(intensity.values),
    "intensity_std_db": np.std(intensity.values),
  }

def pause_features(wav_path, turn_start=None, turn_end=None):
  snd = parselmouth.Sound(wav_path)

  if turn_start is not None and turn_end is not None:
    snd = snd.extract_part(from_time=turn_start, to_time=turn_end)

  duration = snd.duration

  # Praat command: Sound: To TextGrid (silences)...
  # Arguments: min_pitch, time_step, silence_threshold,
  #            min_silent_duration, min_sounding_duration,
  #            silent_label, sounding_label
  textgrid = call(
    snd,
    "To TextGrid (silences)",
    100,          # pitch floor used to derive intensity settings
    0.0,          # time step; 0 = automatic
    -25.0,        # silence threshold relative to maximum intensity, dB
    0.1,          # minimum silence duration, seconds
    0.1,          # minimum sounding duration, seconds
    "pause",      # label for silent intervals
    "speech"      # label for sounding intervals
  )

  tier = call(textgrid, "Get number of intervals", 1)

  pause_duration = 0.0
  n_pauses = 0
  pause_durations = []

  for i in range(1, tier + 1):
    label = call(textgrid, "Get label of interval", 1, i)
    if label == "pause":
      start = call(textgrid, "Get starting point", 1, i)
      end = call(textgrid, "Get end point", 1, i)
      interval_duration = end - start

      pause_duration += interval_duration
      pause_durations.append(interval_duration)
      n_pauses += 1

  speech_duration = duration - pause_duration

  return {
    "file": wav_path,
    "duration_s": duration,
    "pause_duration_s": pause_duration,
    "speech_duration_s": speech_duration,
    "pause_ratio": pause_duration / duration if duration > 0 else np.nan,
    "speech_ratio": speech_duration / duration if duration > 0 else np.nan,
    "n_pauses": n_pauses,
    "mean_pause_duration_s": (
        np.mean(pause_durations) if pause_durations else 0.0
    ),
  }

# Get split speaker files and run analysis for each speaker on all turns at the same time
speaker_filepaths = glob.glob(os.path.join(out_folder, '*', '*.wav'))
files = []
for speaker_filepath in speaker_filepaths:
  files.append(speaker_filepath)
  # patient_id = speaker_filepath.rsplit('/', 1)[0].rsplit('/', 1)[1]
  # print(patient_id, speaker_filepath)

features_F0 = pd.DataFrame([extract_prosodic_feats(f) for f in files])
print(features_F0)
# Save file with dataframe
features_F0.to_csv(os.path.join(out_folder, 'features_diarised_F0.csv'), index=False)

features_pause = pd.DataFrame([pause_features(f) for f in files])
print(features_pause)
# Save file with dataframe
features_pause.to_csv(os.path.join(out_folder, 'features_diarised_pauses.csv'), index=False)


                                                file  duration_s  f0_mean_hz  \
0  /content/audios_split/241219_1309/out_SPEAKER_...  262.911973  103.872493   
1  /content/audios_split/241219_1309/out_SPEAKER_...  131.475986  123.880453   
2  /content/audios_split/250109_1313/out_SPEAKER_...  250.802902  156.901224   
3  /content/audios_split/250109_1313/out_SPEAKER_...  217.603968  123.964202   

   f0_median_hz  f0_std_hz  f0_min_hz   f0_max_hz  f0_range_hz  \
0     93.641684  46.322946  74.936400  594.755741   519.819341   
1    116.866446  32.418987  74.973222  588.412972   513.439751   
2    160.822219  57.541397  75.020311  588.142958   513.122647   
3    116.342674  33.751372  74.954591  579.477527   504.522936   

   voiced_proportion  intensity_mean_db  intensity_std_db  
0           0.525373          61.223313          9.534184  
1           0.604687          58.810474          8.248288  
2           0.470471          58.954472          8.580000  
3           0.546399        

In [6]:
#@title Praat acoustic analysis of full audios (Inter-turn gaps and intra-turn pauses)
# Adapted from Perplexity prompted answers

import parselmouth
import pandas as pd
import numpy as np
from parselmouth.praat import call

# Setup
# sepaker_diarization = 'exclusive'#@param['exclusive', 'non_exclusive']
full_audios_list = glob.glob(os.path.join(audio_folder, '*.wav'))

speaker_feats_all_list = []
turn_feats_all_list = []

for full_audio_path in full_audios_list:
  patient_id = full_audio_path.rsplit('.', 1)[0].rsplit('/', 1)[1]
  output_folder = os.path.join(out_folder, patient_id)
  # Diarized audio files are not used here
  # diarized_files_paths = glob.glob(os.path.join(output_folder, '*.wav'))
  # SPEAKER_AUDIOS = {}
  # for i, diarized_file_path in enumerate(sorted(diarized_files_paths)):
  #   SPEAKER_AUDIOS[f"SPEAKER_{str(i+1).zfill(2)}"] = diarized_file_path
  exclusive_speaker_diarization = pickle.load(open(os.path.join(output_folder, 'diarization_exclusive.pickle'), 'rb'))

  # Praat parameters; may need adjustments
  SILENCE_THRESHOLD_DB = -25.0
  MIN_SILENCE_S = 0.20
  MIN_SOUND_S = 0.10
  EDGE_MARGIN_S = 0.05  # silence ignored at turn edges when computing internal pauses

  # Detect silence once;
  # Run Praat silence detection on the full dialogue, not separately on each speaker WAV. This keeps the same acoustic silence timeline for every turn.
  def get_silence_intervals(wav_path):
    snd = parselmouth.Sound(wav_path)

    textgrid = call(
      snd,
      "To TextGrid (silences)",
      100.0,                    # minimum pitch
      0.0,                      # automatic time step
      SILENCE_THRESHOLD_DB,     # e.g. -25.0
      MIN_SILENCE_S,            # e.g. 0.20
      MIN_SOUND_S,              # e.g. 0.10
      "silence",
      "speech"
    )

    n_intervals = call(textgrid, "Get number of intervals", 1)

    silences = []

    for i in range(1, n_intervals + 1):
      label = call(textgrid, "Get label of interval", 1, i)

      if label == "silence":
        start = call(textgrid, "Get starting point", 1, i)
        end = call(textgrid, "Get end point", 1, i)

        silences.append({
          "start": start,
          "end": end,
          "duration": end - start,
        })

    return silences

  silences = get_silence_intervals(full_audio_path)

  # Build exclusive turns
  # itertracks(yield_label=True) yields (segment, track, speaker_label); the Segment contains start and end.
  def build_turn_table(full_audio_path, exclusive_diarization):
    rows = []

    for turn, _, speaker in exclusive_diarization.itertracks(yield_label=True):
      rows.append({
        "file": full_audio_path,
        "speaker": speaker,
        "turn_start": turn.start,
        "turn_end": turn.end,
        "turn_duration_s": turn.end - turn.start,
      })

    return (
      pd.DataFrame(rows)
      .sort_values("turn_start")
      .reset_index(drop=True)
    )

  turns_df = build_turn_table(full_audio_path, exclusive_speaker_diarization)

  # Pause features per turn
  # This function measures silence inside the turn, while excluding a small margin at each edge. It therefore avoids counting the trailing silence as an internal pause.
  def overlap_duration(interval_start, interval_end, region_start, region_end):
    return max(
      0.0,
      min(interval_end, region_end)
      - max(interval_start, region_start)
    )

  def get_pause_features(row, silences):
    turn_start = row["turn_start"]
    turn_end = row["turn_end"]

    # Region used for internal pauses:
    # excludes a small edge margin from the pyannote turn
    internal_start = turn_start + EDGE_MARGIN_S
    internal_end = turn_end - EDGE_MARGIN_S

    if internal_end <= internal_start:
      internal_start = turn_start
      internal_end = turn_end

    internal_duration = internal_end - internal_start

    internal_silences = []

    for silence in silences:
      duration = overlap_duration(
        silence["start"],
        silence["end"],
        internal_start,
        internal_end
      )

      if duration > 0:
        internal_silences.append(duration)

    pause_duration = sum(internal_silences)

    return {
      "internal_pause_duration_s": pause_duration,
      "internal_pause_ratio": (
        pause_duration / internal_duration
        if internal_duration > 0 else np.nan
      ),
      "n_internal_pauses": len(internal_silences),
      "mean_internal_pause_s": (
        np.mean(internal_silences)
        if internal_silences else 0.0
      ),
      "median_internal_pause_s": (
        np.median(internal_silences)
        if internal_silences else 0.0
      ),
      "longest_internal_pause_s": (
        np.max(internal_silences)
        if internal_silences else 0.0
      ),
    }

  def get_internal_bounds(row):
      turn_start = row["turn_start"]
      turn_end = row["turn_end"]

      internal_start = turn_start + EDGE_MARGIN_S
      internal_end = turn_end - EDGE_MARGIN_S

      if internal_end <= internal_start:
          internal_start = turn_start
          internal_end = turn_end

      return internal_start, internal_end


  def get_internal_duration(row):
      internal_start, internal_end = get_internal_bounds(row)
      return internal_end - internal_start


  def get_internal_pauses(row, silences):
      internal_start, internal_end = get_internal_bounds(row)

      return [
          overlap_duration(
              silence["start"],
              silence["end"],
              internal_start,
              internal_end
          )
          for silence in silences
          if overlap_duration(
              silence["start"],
              silence["end"],
              internal_start,
              internal_end
          ) > 0
      ]

  # Add pause features to dataframe
  def get_internal_pause_duration(row, silences):
    return sum(get_internal_pauses(row, silences))


  def get_n_internal_pauses(row, silences):
    return len(get_internal_pauses(row, silences))


  def get_mean_internal_pause(row, silences):
    pauses = get_internal_pauses(row, silences)
    return np.mean(pauses) if pauses else 0.0


  def get_median_internal_pause(row, silences):
    pauses = get_internal_pauses(row, silences)
    return np.median(pauses) if pauses else 0.0


  def get_longest_internal_pause(row, silences):
    pauses = get_internal_pauses(row, silences)
    return np.max(pauses) if pauses else 0.0

  # pause_features = turns_df.apply(
  #     lambda row: pd.Series(get_pause_features(row, silences)),
  #     axis=1
  # )

  turns_df["internal_pause_duration_s"] = turns_df.apply(
    lambda row: get_internal_pause_duration(row, silences),
    axis=1
  )

  turns_df["internal_pause_ratio"] = turns_df.apply(
    lambda row: (
      row["internal_pause_duration_s"]
      / get_internal_duration(row)
      if get_internal_duration(row) > 0
      else np.nan
    ),
    axis=1
  )

  turns_df["n_internal_pauses"] = turns_df.apply(
    lambda row: get_n_internal_pauses(row, silences),
    axis=1
  )

  turns_df["mean_internal_pause_s"] = turns_df.apply(
    lambda row: get_mean_internal_pause(row, silences),
    axis=1
  )

  turns_df["median_internal_pause_s"] = turns_df.apply(
    lambda row: get_median_internal_pause(row, silences),
    axis=1
  )

  turns_df["longest_internal_pause_s"] = turns_df.apply(
    lambda row: get_longest_internal_pause(row, silences),
    axis=1
  )

  # Trailing silence
  # Trailing silence is the silence between the end of the last sounding region within the turn and the pyannote turn end.
  def get_trailing_silence(row, silences):
    turn_start = row["turn_start"]
    turn_end = row["turn_end"]

    sounding_parts = []

    for silence in silences:
      # Reconstruct sounding intervals as the complement of silence
      pass

    # A simpler approach: find the last silence ending before turn_end
    # and measure from its end to turn_end, if it overlaps the turn.
    relevant_silences = [
      s for s in silences
      if s["end"] > turn_start and s["start"] < turn_end
    ]

    if not relevant_silences:
      return 0.0

    last_silence_end = max(s["end"] for s in relevant_silences)

    return max(0.0, turn_end - last_silence_end)

  turns_df["trailing_silence_s"] = turns_df.apply(
      lambda row: get_trailing_silence(row, silences),
      axis=1
  )

  # Inter-turn gaps and silence
  # Use the end of one exclusive turn and the start of the next. Then measure how much of that gap is acoustically silent.
  turns_df["next_turn_start"] = turns_df["turn_start"].shift(-1)
  turns_df["next_speaker"] = turns_df["speaker"].shift(-1)

  turns_df["inter_turn_gap_s"] = (
      turns_df["next_turn_start"] - turns_df["turn_end"]
  )

  # Last turn has no following turn
  turns_df.loc[turns_df.index[-1], "inter_turn_gap_s"] = np.nan

  turns_df["inter_turn_silence_s"] = turns_df.apply(
    lambda row: overlap_duration(
      row["turn_end"],
      row["next_turn_start"],
      row["turn_end"],
      row["next_turn_start"]
    )
    if pd.notna(row["next_turn_start"])
    else np.nan,
    axis=1
  )

  # More robust: sum silence overlapping the gap
  def silence_in_gap(gap_start, gap_end, silences):
    if pd.isna(gap_start) or pd.isna(gap_end) or gap_end <= gap_start:
      return 0.0

    return sum(
      overlap_duration(
        s["start"],
        s["end"],
        gap_start,
        gap_end
      )
      for s in silences
    )


  turns_df["inter_turn_silence_s"] = turns_df.apply(
    lambda row: silence_in_gap(
      row["turn_end"],
      row["next_turn_start"],
      silences
    ),
    axis=1
  )

  turns_df["inter_turn_silence_ratio"] = turns_df.apply(
    lambda row: (
      row["inter_turn_silence_s"] / row["inter_turn_gap_s"]
      if pd.notna(row["inter_turn_gap_s"])
      and row["inter_turn_gap_s"] > 0
      else np.nan
    ),
    axis=1
  )

  turns_df["transition_type"] = turns_df.apply(
    lambda row: (
      "same_speaker"
      if row["next_speaker"] == row["speaker"]
      else "speaker_change"
    )
    if pd.notna(row["next_speaker"])
    else "end_of_dialogue",
    axis=1
  )

  # Speaker-level profile
  speaker_profile = (
    turns_df
    .groupby(["file", "speaker"])
    .agg(
      n_turns=("turn_duration_s", "count"),
      total_turn_duration_s=("turn_duration_s", "sum"),
      mean_turn_duration_s=("turn_duration_s", "mean"),
      median_turn_duration_s=("turn_duration_s", "median"),

      mean_internal_pause_s=("internal_pause_duration_s", "mean"),
      median_internal_pause_s=("internal_pause_duration_s", "median"),
      mean_internal_pause_ratio=("internal_pause_ratio", "mean"),
      median_internal_pause_ratio=("internal_pause_ratio", "median"),
      mean_n_internal_pauses=("n_internal_pauses", "mean"),
      mean_longest_pause_s=("longest_internal_pause_s", "mean"),

      mean_trailing_silence_s=("trailing_silence_s", "mean"),
      median_trailing_silence_s=("trailing_silence_s", "median"),

      mean_inter_turn_gap_s=("inter_turn_gap_s", "mean"),
      median_inter_turn_gap_s=("inter_turn_gap_s", "median"),

      mean_inter_turn_silence_s=("inter_turn_silence_s", "mean"),
      median_inter_turn_silence_s=("inter_turn_silence_s", "median"),

      mean_inter_turn_silence_ratio=(
          "inter_turn_silence_ratio", "mean"
      ),
      median_inter_turn_silence_ratio=(
          "inter_turn_silence_ratio", "median"
      ),
    )
    .reset_index()
  )

  print(speaker_profile)

  turn_feats_all_list.append(turns_df)
  speaker_feats_all_list.append(speaker_profile)

# Concatenate lists in which dataframes were strored into a dataframe
turn_feats_all = pd.concat(turn_feats_all_list, ignore_index=True)
speaker_feats_all = pd.concat(speaker_feats_all_list, ignore_index=True)

# Save outputs
turn_feats_all.to_csv(os.path.join(out_folder, "features_full_turns.csv"), index=False)
speaker_feats_all.to_csv(os.path.join(out_folder, "features_full_speakers.csv"), index=False)

                                   file     speaker  n_turns  \
0  /content/audios_full/241219_1309.wav  SPEAKER_00       47   
1  /content/audios_full/241219_1309.wav  SPEAKER_01       44   

   total_turn_duration_s  mean_turn_duration_s  median_turn_duration_s  \
0             262.912500              5.593883                2.176875   
1             131.473125              2.988026                1.172812   

   mean_internal_pause_s  median_internal_pause_s  mean_internal_pause_ratio  \
0               2.322187                 0.645875                   0.449949   
1               1.661067                 0.696500                   0.565611   

   median_internal_pause_ratio  mean_n_internal_pauses  mean_longest_pause_s  \
0                     0.417652                4.489362              0.646980   
1                     0.544562                3.659091              0.599298   

   mean_trailing_silence_s  median_trailing_silence_s  mean_inter_turn_gap_s  \
0                  0.1

In [ ]:
#@title Zip and download
!zip -r /content/audios_split.zip /content/audios_split
from google.colab import files
files.download("/content/audios_split.zip")

  adding: content/audios_split/ (stored 0%)
  adding: content/audios_split/features_pause.csv (deflated 51%)
  adding: content/audios_split/241219_1309/ (stored 0%)
  adding: content/audios_split/241219_1309/output_speaker_02.wav (deflated 18%)
  adding: content/audios_split/241219_1309/avg_gaps.json (deflated 25%)
  adding: content/audios_split/241219_1309/log_turns.txt (deflated 80%)
  adding: content/audios_split/241219_1309/output_speaker_01.wav (deflated 20%)
  adding: content/audios_split/250109_1313/ (stored 0%)
  adding: content/audios_split/250109_1313/output_speaker_02.wav (deflated 20%)
  adding: content/audios_split/250109_1313/avg_gaps.json (deflated 16%)
  adding: content/audios_split/250109_1313/log_turns.txt (deflated 81%)
  adding: content/audios_split/250109_1313/output_speaker_01.wav (deflated 21%)
  adding: content/audios_split/241217_1305/ (stored 0%)
  adding: content/audios_split/241217_1305/output_speaker_02.wav (deflated 20%)
  adding: content/audios_split/2412

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>